## Importação de Bibliotecas e Autenticações

In [ ]:
%%capture --no-stderr
%pip install -U langgraph langsmith

# Used for this tutorial; not a requirement for LangGraph
%pip install -U langchain_anthropic
%pip install -U tavily-python
%pip install -U langchain_community
!pip install -U langchain-community tavily-python

In [ ]:
import getpass
import os
import json
from typing import Annotated
from typing import Literal
from typing_extensions import TypedDict
from IPython.display import Image, display

from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import ToolMessage
from langchain_core.messages import BaseMessage
from langchain_core.messages import AIMessage
from langchain_core.pydantic_v1 import BaseModel
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

In [ ]:
_set_env("ANTHROPIC_API_KEY")
_set_env("LANGSMITH_API_KEY")
_set_env("TAVILY_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "LangGraph Tutorial"

ANTHROPIC_API_KEY: ··········
LANGSMITH_API_KEY: ··········
TAVILY_API_KEY: ··········


## Parte 1: Crie um chatbot básico

In [ ]:
class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]

In [ ]:
llm = ChatAnthropic(model="claude-3-haiku-20240307")

In [ ]:
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)
graph = graph_builder.compile()

In [ ]:
while True:
    user_input = input("User: ")
    if user_input.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break
    for event in graph.stream({"messages": ("user", user_input)}):
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)

## Parte 2: Aprimorando o Chatbot com Ferramentas

In [ ]:
class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]

In [ ]:
llm = ChatAnthropic(model="claude-3-haiku-20240307")

In [ ]:
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)
graph = graph_builder.compile()

In [ ]:
class BasicToolNode:
    """Um nó que executa as ferramentas solicitadas na última AIMessage."""
    def __init__(self, tools: list) -> None:
        self.tools_by_name = {tool.name: tool for tool in tools}

    def __call__(self, inputs: dict):
        if messages := inputs.get("messages", []):
            message = messages[-1]
        else:
            raise ValueError("No message found in input")
        outputs = []
        for tool_call in message.tool_calls:
            tool_result = self.tools_by_name[tool_call["name"]].invoke(
                tool_call["args"]
            )
            outputs.append(
                ToolMessage(
                    content=json.dumps(tool_result),
                    name=tool_call["name"],
                    tool_call_id=tool_call["id"],
                )
            )
        return {"messages": outputs}

In [ ]:
tool = TavilySearchResults(max_results=2)
tool_node = BasicToolNode(tools=[tool])
graph_builder.add_node("tools", tool_node)

In [ ]:
def route_tools(
    state: State,
) -> Literal["tools", "__end__"]:
    """
    Use no conditional_edge para rotear para o ToolNode se a última mensagem
    tiver chamadas de ferramenta. Caso contrário, rotear para o final
    """
    if isinstance(state, list):
        ai_message = state[-1]
    elif messages := state.get("messages", []):
        ai_message = messages[-1]
    else:
        raise ValueError(
            f"Nenhuma mensagem encontrada no estado de entrada para tool_edge: {state}")
    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        return "tools"
    return "__end__"

In [ ]:
graph_builder.add_conditional_edges(
    "chatbot",
    route_tools,
    # O dicionário a seguir permite que você diga ao gráfico para interpretar as
    # saídas da condição como um nó específico
    # O padrão é a função de identidade, mas se você
    # deseja usar um nó com outro nome além de "ferramentas",
    # Você pode atualizar o valor do dicionário para outra coisa
    # por exemplo, "ferramentas": "minhas_ferramentas"
    {"tools": "tools", "__end__": "__end__"},
)

# Sempre que uma ferramenta é chamada, voltamos ao chatbot para decidir o próximo passo
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")
graph = graph_builder.compile()

In [ ]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # Isso requer algumas dependências extras e é opcional
    pass

In [ ]:
while True:
    user_input = input("User: ")
    if user_input.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break
    for event in graph.stream({"messages": [("user", user_input)]}):
        for value in event.values():
            if isinstance(value["messages"][-1], BaseMessage):
                print("Assistant:", value["messages"][-1].content)

Assistant: Posso te ajudar com uma ampla variedade de tarefas, desde pesquisas e análises até escrita, resolução de problemas e muito mais. Sou uma assistente virtual com conhecimentos em diversas áreas, como ciências, tecnologia, negócios, criatividade e muito mais. 

Algumas das coisas que posso fazer:

- Ajudar com pesquisas e fornecer informações sobre diversos temas
- Auxiliar na escrita e revisão de textos
- Fazer análises de dados e gerar insights
- Resolver problemas e ajudar a tomar decisões
- Fornecer dicas e sugestões criativas
- Explicar conceitos complexos de forma clara
- Conversar sobre diversos assuntos e trocar ideias

Enfim, estou aqui para te ajudar da melhor forma possível. Basta me dizer o que você precisa e eu farei o meu melhor para te atender. Como posso te ajudar?
User: q


## Parte 3: Adicionando memória ao chatbot

In [ ]:
memory = SqliteSaver.from_conn_string(":memory:")

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
graph_builder = StateGraph(State)


tool = TavilySearchResults(max_results=2)
tools = [tool]
llm = ChatAnthropic(model="claude-3-haiku-20240307")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [ ]:
graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(tools=[tool])
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)
# Any time a tool is called, we return to the chatbot to decide the next step
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

In [ ]:
# compile o gráfico com o ponteiro de verificação fornecido.
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
# Agora você pode interagir com seu bot!
# Primeiro, escolha um tópico para usar como chave para essa conversa

config = {"configurable": {"thread_id": "1"}}

In [ ]:
user_input = "Olá! Meu nome é Ambrosio."

# A configuração é o **segundo argumento posicional** para stream() ou invoke()!
events = graph.stream(
    {"messages": [("user", user_input)]}, config, stream_mode="values"
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Olá! Meu nome é Ambrosio.
================================== Ai Message ==================================

Olá Ambrosio, é um prazer conhecê-lo! Espero que possamos ter uma ótima conversa. Como posso ajudá-lo hoje?


In [ ]:
# user_input = "Lembra do meu nome?"
user_input = """
              este é todo código necessário?
              """
# A configuração é o **segundo argumento posicional** para stream() ou invoke()!
events = graph.stream(
    {"messages": [("user", user_input)]}, config, stream_mode="values" )

for event in events: event["messages"][-1].pretty_print()

================================ Human Message =================================


              este é todo código necessário?
              
================================== Ai Message ==================================

Não, o código que eu mostrei anteriormente é apenas uma parte do que é necessário para implementar um chatbot usando LangChain e integrar com um banco de dados. Vou detalhar o código completo abaixo:

```python
from langchain.llms import OpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferWindowMemory
from langchain.agents import initialize_agent, Tool
from langchain.utilities import SQLDatabaseWrapper
from langchain.prompts.prompt import PromptTemplate

# Configurar o modelo de linguagem
llm = OpenAI(temperature=0.7)

# Conectar ao banco de dados
db = SQLDatabaseWrapper('sqlite:///products.db')

# Criar as ferramentas para o agente
tools = [
    Tool(
        name="Product Lookup",
        func=lambda query: db.run(

In [ ]:
# A única diferença é que alteramos o `thread_id` aqui para "2" em vez de "1"
events = graph.stream( {"messages": [("user", user_input)]}, {"configurable": {"thread_id": "2"}}, stream_mode="values", )

for event in events:
  event["messages"][-1].pretty_print()

================================ Human Message =================================

Lembra do meu nome?
================================== Ai Message ==================================

Infelizmente, não me lembro do seu nome. Como assistente virtual, não tenho a capacidade de lembrar detalhes específicos sobre usuários individuais. Mas fico feliz em conversar consigo e ajudá-lo com o que precisar! Por favor, sinta-se à vontade para me dizer seu nome novamente, caso queira que eu me refira a você por ele.


In [ ]:
# Para inspecionar um gráfico statepara uma configuração dada a qualquer momento, chame get_state(config).
snapshot = graph.get_state(config)
snapshot

StateSnapshot(values={'messages': [HumanMessage(content='Olá! Meu nome é Ambrosio.', id='234095d6-5ddf-4b26-b4cd-15a7785edb4e'), AIMessage(content=[{'id': 'toolu_0111FCBFTHsNHqfj41eZLBFj', 'input': {'query': 'Ambrosio'}, 'name': 'tavily_search_results_json', 'type': 'tool_use'}], response_metadata={'id': 'msg_01GzUssooXsYLgAjCZPKtLF5', 'model': 'claude-3-haiku-20240307', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'input_tokens': 381, 'output_tokens': 61}}, id='run-1810ceb8-d22e-44ee-8d51-c5ee2f1f3aed-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Ambrosio'}, 'id': 'toolu_0111FCBFTHsNHqfj41eZLBFj', 'type': 'tool_call'}], usage_metadata={'input_tokens': 381, 'output_tokens': 61, 'total_tokens': 442}), ToolMessage(content='[{"url": "https://en.wikipedia.org/wiki/Alessandra_Ambrosio", "content": "Alessandra Ambrosio. Alessandra Corine Ambr\\u00f3sio [5] ( Brazilian Portuguese: [ale\\u02c8s\\u0250\\u0303d\\u027e\\u00e3\\u02c8b\\u027e\\u0254zju]; b

In [ ]:
"""
como o gráfico terminou neste turno, `next` está vazio.
Se você buscar um estado de dentro de uma invocação de gráfico,
next informa qual nó será executado em seguida
"""
snapshot.next

()

## Parte 4: Humano no circuito

Os agentes podem não ser confiáveis ​​e podem precisar de entrada humana para realizar tarefas com sucesso. Da mesma forma, para algumas ações, você pode querer exigir aprovação humana antes de executar para garantir que tudo esteja funcionando conforme o esperado.

O LangGraph suporta human-in-the-loopfluxos de trabalho de várias maneiras. Nesta seção, usaremos interrupt_beforea funcionalidade do LangGraph para sempre quebrar o nó da ferramenta.

Primeiro, comece com nosso código existente. O seguinte é copiado da Parte 3.

In [ ]:
memory = SqliteSaver.from_conn_string(":memory:")

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
graph_builder = StateGraph(State)


tool = TavilySearchResults(max_results=2)
tools = [tool]
llm = ChatAnthropic(model="claude-3-haiku-20240307")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [ ]:
graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(tools=[tool])
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")
graph  =  graph_builder . compile (
    checkpointer = memory ,
    # Isso é novo!
    interrupt_before = [ "tools" ],
    # Nota: também pode interromper ações __after__, se desejado.
    # interrupt_after=["tools"]
)

In [ ]:
user_input  =  "Estou aprendendo LangGraph. Você poderia pesquisar um pouco sobre isso para mim?"
config = {"configurable": {"thread_id": "1"}}
# The config is the **second positional argument** to stream() or invoke()!
events = graph.stream(
    {"messages": [("user", user_input)]}, config, stream_mode="values"
)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

Estou aprendendo LangGraph. Você poderia pesquisar um pouco sobre isso para mim?
================================== Ai Message ==================================

[{'text': 'Claro, vou fazer uma pesquisa sobre LangGraph para você.', 'type': 'text'}, {'id': 'toolu_01UZiwg1CBv5bpiaGFoBDB3B', 'input': {'query': 'LangGraph'}, 'name': 'tavily_search_results_json', 'type': 'tool_use'}]
Tool Calls:
  tavily_search_results_json (toolu_01UZiwg1CBv5bpiaGFoBDB3B)
 Call ID: toolu_01UZiwg1CBv5bpiaGFoBDB3B
  Args:
    query: LangGraph


In [ ]:
snapshot = graph.get_state(config)
snapshot.next

('tools',)

In [ ]:
existing_message = snapshot.values["messages"][-1]
existing_message.tool_calls

[{'name': 'tavily_search_results_json',
  'args': {'query': 'LangGraph'},
  'id': 'toolu_01UZiwg1CBv5bpiaGFoBDB3B',
  'type': 'tool_call'}]

In [ ]:
# None não acrescentará nada de novo ao estado atual,
# permitindo que ele continue como se nunca tivesse sido interrompido
events = graph.stream(None, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================= Tool Message =================================
Name: tavily_search_results_json

[{"url": "https://www.langchain.com/langgraph", "content": "LangGraph sets the foundation for how we can build and scale AI workloads \u2014 from conversational agents, complex task automation, to custom LLM-backed experiences that 'just work'. The next chapter in building complex production-ready features with LLMs is agentic, and with LangGraph and LangSmith, LangChain delivers an out-of-the-box solution ..."}, {"url": "https://langchain-ai.github.io/langgraph/", "content": "LangGraph is a framework for creating stateful, multi-actor applications with LLMs, using cycles, controllability, and persistence. It integrates with LangChain and LangSmith, and supports streaming, human-in-the-loop, and memory features."}]
================================== Ai Message ==================================

Os resultados da pesquisa indicam que o LangGraph é uma estrutura (framework) 

## Parte 5: Atualizando manualmente o estado

Na seção anterior, mostramos como interromper um gráfico para que um humano possa inspecionar suas ações. Isso permite que o humano veja read o estado, mas se ele quiser mudar o curso do agente, precisará ter write acesso.

Felizmente, o LangGraph permite que você atualize manualmente o estado ! Atualizar o estado permite que você controle a trajetória do agente modificando suas ações (até mesmo modificando o passado!). Esse recurso é particularmente útil quando você quer corrigir os erros do agente, explorar caminhos alternativos ou guiar o agente em direção a um objetivo específico.

Mostraremos como atualizar um estado de checkpoint abaixo. Como antes, primeiro, defina seu gráfico. Reutilizaremos exatamente o mesmo gráfico de antes.

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
graph_builder = StateGraph(State)


tool = TavilySearchResults(max_results=2)
tools = [tool]
llm = ChatAnthropic(model="claude-3-haiku-20240307")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [ ]:
graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(tools=[tool])
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")
memory = SqliteSaver.from_conn_string(":memory:")
graph = graph_builder.compile(
    checkpointer=memory,
    # This is new!
    interrupt_before=["tools"],
    # Note: can also interrupt **after** actions, if desired.
    # interrupt_after=["tools"]
)

In [ ]:
user_input = "Estou aprendendo LangGraph. Você poderia fazer alguma pesquisa sobre isso para mim?"
config = {"configurable": {"thread_id": "1"}}
# The config is the **second positional argument** to stream() or invoke()!
events = graph.stream({"messages": [("user", user_input)]}, config)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

In [ ]:
snapshot = graph.get_state(config)
existing_message = snapshot.values["messages"][-1]
existing_message.pretty_print()

================================== Ai Message ==================================

[{'id': 'toolu_01A7nbSb7Wt4LXw6JJ16PnMV', 'input': {'query': 'LangGraph'}, 'name': 'tavily_search_results_json', 'type': 'tool_use'}]
Tool Calls:
  tavily_search_results_json (toolu_01A7nbSb7Wt4LXw6JJ16PnMV)
 Call ID: toolu_01A7nbSb7Wt4LXw6JJ16PnMV
  Args:
    query: LangGraph


Até agora, tudo isso é uma repetição exata da seção anterior. O LLM apenas solicitou o uso da ferramenta de mecanismo de busca e nosso gráfico foi interrompido. Se prosseguirmos como antes, a ferramenta será chamada para pesquisar na web.

Mas e se o usuário quiser interceder? E se acharmos que o chatbot não precisa usar a ferramenta?

Vamos dar a resposta correta diretamente!

In [ ]:
answer  =  (
    "LangGraph é uma biblioteca para construir aplicativos multiator com estado com LLMs."
)
new_messages  =  [
    # A API LLM espera que alguma ToolMessage corresponda à sua chamada de ferramenta.
    # Nós iremos satisfazer isso aqui.
    ToolMessage ( content = answer ,  tool_call_id = existing_message . tool_calls [ 0 ][ "id" ]),
    # E então diretamente "colocar palavras na boca do LLM" preenchendo sua resposta.
    AIMessage ( content = answer ),
]

In [ ]:
new_messages[-1].pretty_print()
graph.update_state(
    # Qual estado atualizar
    config,
    # Os valores atualizados a serem fornecidos. As mensagens em nosso `State`
    # são "somente anexadas", o que significa que serão anexadas
    # para o estado existente. Analisaremos como atualizar mensagens
    # existentes na próxima seção!
    {"messages": new_messages},
)
print("\n\nLast 2 messages;")
print(graph.get_state(config).values["messages"][-2:])

================================== Ai Message ==================================

LangGraph é uma biblioteca para construir aplicativos multiator com estado com LLMs.


Last 2 messages;
[ToolMessage(content='LangGraph é uma biblioteca para construir aplicativos multiator com estado com LLMs.', id='9222d49b-ec01-4107-913f-c6a3e65dc2fa', tool_call_id='toolu_01GMqphStEccvXvTHv9h4pfF'), AIMessage(content='LangGraph é uma biblioteca para construir aplicativos multiator com estado com LLMs.', id='70b6bc90-16b9-4adc-bd60-d5963667721d')]


A função update_state opera como se fosse um dos nós do seu gráfico! Por padrão, a operação de atualização usa o nó que foi executado por último, mas você pode especificá-lo manualmente abaixo. Vamos adicionar uma atualização e dizer ao gráfico para tratá-la como se viesse do "chatbot".

In [ ]:
graph.update_state(
    config,
    {"messages": [AIMessage(content="Sou um especialista em IA!")]},
    # Qual nó para esta função atuar. Ele continuará automaticamente
    # processando como se este nó tivesse acabado de ser executado.
    as_node="chatbot",
)

{'configurable': {'thread_id': '1',
  'thread_ts': '1ef42357-2536-6f35-8002-cae779d59dbd'}}

In [ ]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
snapshot = graph.get_state(config)
print(snapshot.values["messages"][-3:])
print(snapshot.next)

[HumanMessage(content='Estou aprendendo LangGraph. Você poderia fazer alguma pesquisa sobre isso para mim?', id='5c34befe-8ca2-404e-b295-e0b34be85268'), AIMessage(content=[{'id': 'toolu_01A7nbSb7Wt4LXw6JJ16PnMV', 'input': {'query': 'LangGraph'}, 'name': 'tavily_search_results_json', 'type': 'tool_use'}], response_metadata={'id': 'msg_01KceFDJdntNsshNex1brEpq', 'model': 'claude-3-haiku-20240307', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'input_tokens': 393, 'output_tokens': 60}}, id='run-a807de10-d1d1-4564-9a04-1049046df0ad-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'LangGraph'}, 'id': 'toolu_01A7nbSb7Wt4LXw6JJ16PnMV', 'type': 'tool_call'}], usage_metadata={'input_tokens': 393, 'output_tokens': 60, 'total_tokens': 453}), AIMessage(content='Sou um especialista em IA!', id='9d730c04-2d63-4212-bd8e-e133a906fa88')]
()


Aviso : continuamos adicionando mensagens de IA ao estado. Como estamos agindo como chatbot respondendo com uma AIMessage que não contém tool_calls, o gráfico sabe que entrou em um estado finalizado ( next está vazio).

E se você quiser substituir mensagens existentes?
A add_messages função que usamos para anotar nossos gráficos State acima controla como as atualizações são feitas na messages chave. Esta função olha para quaisquer IDs de mensagem na nova messages lista. Se o ID corresponder a uma mensagem no estado existente, add_messages sobrescreve a mensagem existente com o novo conteúdo.

Como exemplo, vamos atualizar a invocação da ferramenta para garantir que tenhamos bons resultados do nosso mecanismo de busca! Primeiro, inicie um novo thread:

In [ ]:
user_input = "Estou aprendendo LangGraph. Você poderia pesquisar um pouco sobre ele para mim?"
config = {"configurable": {"thread_id": "2"}}
# usaremos thread_id = 2 aqui
events = graph.stream( {"messages": [("user", user_input)]}, config, stream_mode="values" )
for event in events:
  if "messages" in event:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Estou aprendendo LangGraph. Você poderia pesquisar um pouco sobre ele para mim?
================================== Ai Message ==================================

[{'id': 'toolu_01Kin1KyJVwtpAiH8w9uXzXQ', 'input': {'query': 'langraph'}, 'name': 'tavily_search_results_json', 'type': 'tool_use'}]
Tool Calls:
  tavily_search_results_json (toolu_01Kin1KyJVwtpAiH8w9uXzXQ)
 Call ID: toolu_01Kin1KyJVwtpAiH8w9uXzXQ
  Args:
    query: langraph


Em seguida, vamos atualizar a invocação da ferramenta para nosso agente. Talvez queiramos procurar por fluxos de trabalho de humano no loop em particular.

In [ ]:
snapshot = graph.get_state(config)
existing_message = snapshot.values["messages"][-1]
print("Original")
print("Message ID", existing_message.id)
print(existing_message.tool_calls[0])
new_tool_call = existing_message.tool_calls[0].copy()
new_tool_call["args"]["query"] = "LangGraph human-in-the-loop workflow"
new_message = AIMessage(
    content=existing_message.content,
    tool_calls=[new_tool_call],
    # Important! The ID is how LangGraph knows to REPLACE the message in the state rather than APPEND this messages
    id=existing_message.id,
)

print("Updated")
print(new_message.tool_calls[0])
print("Message ID", new_message.id)
graph.update_state(config, {"messages": [new_message]})

print("\n\nTool calls")
graph.get_state(config).values["messages"][-1].tool_calls

Original
Message ID run-988b6e7d-dad7-483a-949d-4ebce7bf28c0-0
{'name': 'tavily_search_results_json', 'args': {'query': 'langraph'}, 'id': 'toolu_01Kin1KyJVwtpAiH8w9uXzXQ', 'type': 'tool_call'}
Updated
{'name': 'tavily_search_results_json', 'args': {'query': 'LangGraph human-in-the-loop workflow'}, 'id': 'toolu_01Kin1KyJVwtpAiH8w9uXzXQ', 'type': 'tool_call'}
Message ID run-988b6e7d-dad7-483a-949d-4ebce7bf28c0-0


Tool calls


[{'name': 'tavily_search_results_json',
  'args': {'query': 'LangGraph human-in-the-loop workflow'},
  'id': 'toolu_01Kin1KyJVwtpAiH8w9uXzXQ',
  'type': 'tool_call'}]

Observe que modificamos a invocação da ferramenta de IA para procurar por "fluxo de trabalho humano no loop do LangGraph" em vez do simples "LangGraph".

Confira o rastreamento do LangSmith para ver a chamada de atualização de estado - você pode ver que nossa nova mensagem atualizou com sucesso a mensagem de IA anterior.

Retome o gráfico transmitindo com uma entrada None e a configuração existente.

In [ ]:
events = graph.stream(None, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================= Tool Message =================================
Name: tavily_search_results_json

[{"url": "https://www.youtube.com/watch?v=9BPCV5TYPmg", "content": "In this video, I'll show you how to handle persistence with LangGraph, enabling a unique Human-in-the-Loop workflow. This approach allows a human to grant an..."}, {"url": "https://github.com/langchain-ai/langgraph/blob/main/examples/human-in-the-loop.ipynb", "content": "Contribute to langchain-ai/langgraph development by creating an account on GitHub. ... Product Actions. Automate any workflow Packages. Host and manage packages Security. Find and fix vulnerabilities Codespaces. Instant dev environments GitHub Copilot ... human-in-the-loop.ipynb. Top. File metadata and controls. Preview. Code. Blame. 675 lines ..."}]
================================== Ai Message ==================================

Based on the search results, LangGraph is a tool that enables a human-in-the-loop workflow. This allows a huma

In [ ]:
events = graph.stream(
    {
        "messages": (
            "user",
            "Remember what I'm learning about?",
        )
    },
    config,
    stream_mode="values",
)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

Remember what I'm learning about?
================================== Ai Message ==================================

Ah yes, you mentioned that you are learning LangGraph. The search results I provided were about the human-in-the-loop workflow capabilities of LangGraph, which seems to be a key feature of the tool that you are currently studying.

Let me summarize what I've learned about LangGraph in the context of your learning:

- LangGraph is a tool that you are currently learning about and exploring.
- It enables a human-in-the-loop workflow, where a human can interact with and guide an AI system during its operations.
- This allows for persistence of state and data between interactions, enabling a continuous, iterative workflow.
- The human can directly influence the AI system's actions and decision-making process, working collaboratively with the AI.
- LangGraph appears to be designed to integrate wit

## Parte 6: Personalizando o estado

té agora, confiamos em um estado simples (é apenas uma lista de mensagens!). Você pode ir longe com esse estado simples, mas se quiser definir um comportamento complexo sem depender da lista de mensagens, você pode adicionar campos adicionais ao estado. Nesta seção, estenderemos nosso bot de bate-papo com um novo nó para ilustrar isso.

Nos exemplos acima, envolvemos um humano deterministicamente: o gráfico sempre interrompia quando uma ferramenta era invocada. Suponha que quiséssemos que nosso chatbot tivesse a opção de confiar em um humano.

Uma maneira de fazer isso é criar um nó "humano" de passagem, antes do qual o gráfico sempre parará. Só executaremos esse nó se o LLM invocar uma ferramenta "humana". Para nossa conveniência, incluiremos um sinalizador "ask_human" em nosso estado de gráfico que inverteremos se o LLM chamar essa ferramenta.

Abaixo, defina este novo gráfico, com uma atualização State

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    # This flag is new
    ask_human: bool

In [ ]:
# defina um esquema para mostrar ao modelo e permitir que ele decida solicitar assistência
class RequestAssistance(BaseModel):
  """Encaminhe a conversa para um especialista. Use isso se você não puder
  ajudar diretamente ou se o usuário precisar de suporte além de suas permissões.
  Para usar esta função, retransmita a 'solicitação' do usuário para que o
  especialista possa fornecer a orientação correta. """
  request: str

Em seguida, defina o nó do chatbot. A modificação primária aqui é inverter o ask_human sinalizador se virmos que o chatbot invocou o Request Assistance sinalizador.

In [ ]:
tool = TavilySearchResults(max_results=2)
tools = [tool]
llm = ChatAnthropic(model="claude-3-haiku-20240307")
# We can bind the llm to a tool definition, a pydantic model, or a json schema
llm_with_tools = llm.bind_tools(tools + [RequestAssistance])


def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    ask_human = False
    if (
        response.tool_calls
        and response.tool_calls[0]["name"] == RequestAssistance.__name__
    ):
        ask_human = True
    return {"messages": [response], "ask_human": ask_human}


Em seguida, crie o construtor de gráficos e adicione os nós de chatbot e ferramentas ao gráfico, como antes.

In [ ]:
graph_builder = StateGraph(State)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=[tool]))

Em seguida, crie o "humano" node. Esta node função é principalmente um espaço reservado em nosso gráfico que acionará uma interrupção. Se o humano não atualizar manualmente o estado durante o interrupt, ele insere uma mensagem de ferramenta para que o LLM saiba que o usuário foi solicitado, mas não respondeu. Este nó também desmarca o ask_human sinalizador para que o gráfico saiba que não deve revisitar o nó, a menos que outras solicitações sejam feitas.

In [ ]:
def create_response(response: str, ai_message: AIMessage):
    return ToolMessage(
        content=response,
        tool_call_id=ai_message.tool_calls[0]["id"],
    )


def human_node(state: State):
    new_messages = []
    if not isinstance(state["messages"][-1], ToolMessage):
        # Typically, the user will have updated the state during the interrupt.
        # If they choose not to, we will include a placeholder ToolMessage to
        # let the LLM continue.
        new_messages.append(
            create_response("No response from human.", state["messages"][-1])
        )
    return {
        # Append the new messages
        "messages": new_messages,
        # Unset the flag
        "ask_human": False,
    }


graph_builder.add_node("human", human_node)

Em seguida, defina a lógica condicional. Ela select_next_node roteará para o human nó se o sinalizador estiver definido. Caso contrário, ela permite que a tools_condition função pré-construída escolha o próximo nó.

Lembre-se de que a tools_condition função simplesmente verifica se o chatbot respondeu com any tool_calls em sua mensagem de resposta. Se sim, ele roteia para o action nó. Caso contrário, ele encerra o gráfico.

In [ ]:
def select_next_node(state: State):
    if state["ask_human"]:
        return "human"
    # Otherwise, we can route as before
    return tools_condition(state)


graph_builder.add_conditional_edges(
    "chatbot",
    select_next_node,
    {"human": "human", "tools": "tools", "__end__": "__end__"},
)

Por fim, adicione as arestas direcionadas simples e compile o gráfico. Essas arestas instruem o gráfico a sempre fluir de node a -> b sempre a termina a execução.

In [ ]:
# The rest is the same
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("human", "chatbot")
graph_builder.add_edge(START, "chatbot")
memory = SqliteSaver.from_conn_string(":memory:")
graph = graph_builder.compile(
    checkpointer=memory,
    # We interrupt before 'human' here instead.
    interrupt_before=["human"],
)

Se você tiver as dependências de visualização instaladas, poderá ver a estrutura do gráfico abaixo:

In [ ]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

O chatbot pode solicitar ajuda de um humano (chatbot->select->human), invocar a ferramenta do mecanismo de busca (chatbot->select->action) ou responder diretamente (chatbot->select-> end). Uma vez que uma ação ou solicitação tenha sido feita, o gráfico fará a transição de volta para o chatbot nó para continuar as operações.

Vamos ver este gráfico em ação. Solicitaremos assistência especializada para ilustrar nosso gráfico.

In [ ]:
user_input = "Preciso de orientação especializada para construir este agente de IA. Você poderia solicitar assistência para mim?"
config = {"configurable": {"thread_id": "1"}}
# The config is the **second positional argument** to stream() or invoke()!
events = graph.stream(
    {"messages": [("user", user_input)]}, config, stream_mode="values"
)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

Preciso de orientação especializada para construir este agente de IA. Você poderia solicitar assistência para mim?
================================== Ai Message ==================================

[{'id': 'toolu_011qXwgdGrmBUZkEdM4tufuD', 'input': {'request': 'Preciso de orientação especializada para construir este agente de IA.'}, 'name': 'RequestAssistance', 'type': 'tool_use'}]
Tool Calls:
  RequestAssistance (toolu_011qXwgdGrmBUZkEdM4tufuD)
 Call ID: toolu_011qXwgdGrmBUZkEdM4tufuD
  Args:
    request: Preciso de orientação especializada para construir este agente de IA.


Aviso: o LLM invocou a Request Assistance ferramenta " " que fornecemos a ele, e a interrupção foi definida. Vamos inspecionar o estado do gráfico para confirmar.

In [ ]:
snapshot = graph.get_state(config)
snapshot.next

('human',)

O estado do gráfico é de fato interrompido antes do 'human'nó. Podemos agir como o "especialista" neste cenário e atualizar manualmente o estado adicionando uma nova ToolMessage com nossa entrada.

Em seguida, responda à solicitação do chatbot:

Criando um ToolMessagecom nessa resposta. Isso será passado de volta para o chatbot.
Chamada update_state para atualizar manualmente o estado do gráfico.

In [ ]:
ai_message = snapshot.values["messages"][-1]
human_response = (
    """Nós, os especialistas, estamos aqui para ajudar!
     Recomendamos que você verifique o LangGraph para construir seu agente.
    É muito mais confiável e extensível do que simples agentes autônomos."""
)
tool_message = create_response(human_response, ai_message)
graph.update_state(config, {"messages": [tool_message]})

{'configurable': {'thread_id': '1',
  'thread_ts': '1ef423a3-2d35-6d62-8002-e9552ec43614'}}


Você pode inspecionar o estado para confirmar se nossa resposta foi adicionada.

In [ ]:
graph.get_state(config).values["messages"]

[HumanMessage(content='Preciso de orientação especializada para construir este agente de IA. Você poderia solicitar assistência para mim?', id='612fa8de-074e-4fa2-b2b9-f6c133968c1f'),
 AIMessage(content=[{'id': 'toolu_011qXwgdGrmBUZkEdM4tufuD', 'input': {'request': 'Preciso de orientação especializada para construir este agente de IA.'}, 'name': 'RequestAssistance', 'type': 'tool_use'}], response_metadata={'id': 'msg_014kWCbxJhA1Kd9qc6DxdggQ', 'model': 'claude-3-haiku-20240307', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'input_tokens': 595, 'output_tokens': 70}}, id='run-6eb63203-557c-4558-99c0-8a568a609f73-0', tool_calls=[{'name': 'RequestAssistance', 'args': {'request': 'Preciso de orientação especializada para construir este agente de IA.'}, 'id': 'toolu_011qXwgdGrmBUZkEdM4tufuD', 'type': 'tool_call'}], usage_metadata={'input_tokens': 595, 'output_tokens': 70, 'total_tokens': 665}),
 ToolMessage(content='Nós, os especialistas, estamos aqui para ajudar! \n     Recom

In [ ]:
events = graph.stream(None, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================= Tool Message =================================

Nós, os especialistas, estamos aqui para ajudar! 
     Recomendamos que você verifique o LangGraph para construir seu agente.
    É muito mais confiável e extensível do que simples agentes autônomos.
================================== Ai Message ==================================

Encaminhei sua solicitação de orientação especializada para construir o agente de IA. Os especialistas estão disponíveis para fornecer orientação mais detalhada sobre como projetar e desenvolver seu agente de IA de forma eficaz. Eles recomendaram verificar o LangGraph, que é uma estrutura confiável e extensível para construir agentes de IA. Portanto, estão prontos para lhe dar instruções mais específicas assim que necessário.


## Parte 7: Viagem no tempo

Em um fluxo de trabalho típico de bot de bate-papo, o usuário interage com o bot 1 ou mais vezes para realizar uma tarefa. Nas seções anteriores, vimos como adicionar memória e um humano no loop para poder verificar o estado do nosso gráfico e substituir manualmente o estado para controlar respostas futuras.

Mas e se você quiser deixar seu usuário começar de uma resposta anterior e "ramificar" para explorar um resultado separado? Ou se você quiser que os usuários possam "rebobinar" o trabalho do seu assistente para corrigir alguns erros ou tentar uma estratégia diferente (comum em aplicativos como engenheiros de software autônomos)?

Você pode criar essas duas experiências e muito mais usando a funcionalidade integrada de "viagem no tempo" do LangGraph.

Nesta seção, você "rebobinará" seu gráfico buscando um ponto de verificação usando o get_state_history método do gráfico. Você pode então retomar a execução neste ponto anterior no tempo.

Primeiro, lembre-se do nosso gráfico de chatbot. Não precisamos fazer nenhuma alteração de antes:

In [ ]:
class RequestAssistance(BaseModel):
    """Escalate the conversation to an expert. Use this if you are unable to assist directly or if the user requires support beyond your permissions.

    To use this function, relay the user's 'request' so the expert can provide the right guidance.
    """

    request: str


tool = TavilySearchResults(max_results=2)
tools = [tool]
llm = ChatAnthropic(model="claude-3-haiku-20240307")
# We can bind the llm to a tool definition, a pydantic model, or a json schema
llm_with_tools = llm.bind_tools(tools + [RequestAssistance])


def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    ask_human = False
    if (
        response.tool_calls
        and response.tool_calls[0]["name"] == RequestAssistance.__name__
    ):
        ask_human = True
    return {"messages": [response], "ask_human": ask_human}


graph_builder = StateGraph(State)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=[tool]))


def create_response(response: str, ai_message: AIMessage):
    return ToolMessage(
        content=response,
        tool_call_id=ai_message.tool_calls[0]["id"],
    )


def human_node(state: State):
    new_messages = []
    if not isinstance(state["messages"][-1], ToolMessage):
        # Typically, the user will have updated the state during the interrupt.
        # If they choose not to, we will include a placeholder ToolMessage to
        # let the LLM continue.
        new_messages.append(
            create_response("No response from human.", state["messages"][-1])
        )
    return {
        # Append the new messages
        "messages": new_messages,
        # Unset the flag
        "ask_human": False,
    }


graph_builder.add_node("human", human_node)


def select_next_node(state: State) -> Literal["human", "tools", "__end__"]:
    if state["ask_human"]:
        return "human"
    # Otherwise, we can route as before
    return tools_condition(state)


graph_builder.add_conditional_edges(
    "chatbot",
    select_next_node,
    {"human": "human", "tools": "tools", "__end__": "__end__"},
)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("human", "chatbot")
graph_builder.add_edge(START, "chatbot")
memory = SqliteSaver.from_conn_string(":memory:")
graph = graph_builder.compile(
    checkpointer=memory,
    interrupt_before=["human"],
)

Vamos fazer nosso gráfico dar alguns passos. Cada passo será marcado em seu histórico de estado:

In [ ]:
config = {"configurable": {"thread_id": "1"}}
events = graph.stream(
    {
        "messages": [
            ("user","Estou aprendendo LangGraph. Você poderia fazer uma pesquisa sobre isso para mim?")
        ]
    },
    config,
    stream_mode="values",
)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

Estou aprendendo LangGraph. Você poderia fazer uma pesquisa sobre isso para mim?
================================== Ai Message ==================================

[{'id': 'toolu_01G7S2Aoj6TemURixsNpySC3', 'input': {'query': 'LangGraph'}, 'name': 'tavily_search_results_json', 'type': 'tool_use'}]
Tool Calls:
  tavily_search_results_json (toolu_01G7S2Aoj6TemURixsNpySC3)
 Call ID: toolu_01G7S2Aoj6TemURixsNpySC3
  Args:
    query: LangGraph
================================= Tool Message =================================
Name: tavily_search_results_json

[{"url": "https://www.langchain.com/langgraph", "content": "LangGraph sets the foundation for how we can build and scale AI workloads \u2014 from conversational agents, complex task automation, to custom LLM-backed experiences that 'just work'. The next chapter in building complex production-ready features with LLMs is agentic, and with LangGraph and LangSmith

In [ ]:
events = graph.stream(
    {
        "messages": [
            ("user", "Sim, isso é útil. Talvez eu construa um agente autônomo com isso!")
        ]
    },
    config,
    stream_mode="values",
)

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

Sim, isso é útil. Talvez eu construa um agente autônomo com isso!
================================== Ai Message ==================================

[{'text': 'Excelente, construir um agente autônomo com o LangGraph parece uma ótima aplicação dessa ferramenta! Algumas considerações adicionais:', 'type': 'text'}, {'id': 'toolu_01XGb7JsFxNcc8dTCEpy49BF', 'input': {'query': 'autonomous agent with LangGraph'}, 'name': 'tavily_search_results_json', 'type': 'tool_use'}]
Tool Calls:
  tavily_search_results_json (toolu_01XGb7JsFxNcc8dTCEpy49BF)
 Call ID: toolu_01XGb7JsFxNcc8dTCEpy49BF
  Args:
    query: autonomous agent with LangGraph
================================= Tool Message =================================
Name: tavily_search_results_json

[{"url": "https://blog.langchain.dev/how-to-build-the-ultimate-ai-automation-with-multi-agent-collaboration/", "content": "This is the \"master\" agent that coordinates 

Agora que o agente executou algumas etapas, podemos obter replay o histórico completo do estado para ver tudo o que ocorreu.

In [ ]:
to_replay = None
for state in graph.get_state_history(config):
    print("Num Messages: ", len(state.values["messages"]), "Next: ", state.next)
    print("-" * 80)
    if len(state.values["messages"]) == 6:
        # We are somewhat arbitrarily selecting a specific state based on the number of chat messages in the state.
        to_replay = state

Num Messages:  12 Next:  ()
--------------------------------------------------------------------------------
Num Messages:  11 Next:  ('chatbot',)
--------------------------------------------------------------------------------
Num Messages:  10 Next:  ('tools',)
--------------------------------------------------------------------------------
Num Messages:  9 Next:  ('chatbot',)
--------------------------------------------------------------------------------
Num Messages:  8 Next:  ('__start__',)
--------------------------------------------------------------------------------
Num Messages:  8 Next:  ()
--------------------------------------------------------------------------------
Num Messages:  7 Next:  ('chatbot',)
--------------------------------------------------------------------------------
Num Messages:  6 Next:  ('tools',)
--------------------------------------------------------------------------------
Num Messages:  5 Next:  ('chatbot',)
--------------------------------------

Observe que os checkpoints são salvos para cada etapa do gráfico. Isso abrange invocações para que você possa retroceder no histórico completo de um thread. Escolhemos to_replaycomo um estado para retomar. Este é o estado após o chatbotnó na segunda invocação do gráfico acima.

Continuar a partir deste ponto deve chamar o próximo nó de ação

In [ ]:
print(to_replay.next)
print(to_replay.config)

('tools',)
{'configurable': {'thread_id': '1', 'thread_ts': '1ef423b1-7056-601f-8006-4391b615bf57'}}


Observe que o config ( ) do check point to_replay. config contém um thread_ts timestamp . Fornecer esse thread_ts valor informa ao check pointer do LangGraph para carregar o estado daquele momento no tempo. Vamos tentar abaixo:

In [ ]:
# The `thread_ts` in the `to_replay.config` corresponds to a state we've persisted to our checkpointer.
for event in graph.stream(None, to_replay.config, stream_mode="values"):
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================= Tool Message =================================
Name: tavily_search_results_json

[{"url": "https://www.langchain.com/langgraph", "content": "LangGraph is a framework for building agents that handle complex scenarios with LLMs. It supports diverse control flows, human-agent collaboration, streaming, and deployment at scale."}, {"url": "https://langchain-ai.github.io/langgraph/", "content": "LangGraph is a framework for creating stateful, multi-actor applications with LLMs, using cycles, controllability, and persistence. Learn how to use LangGraph to build agent and multi-agent workflows with examples, features, and integration with LangChain."}]
================================== Ai Message ==================================

Com base nos resultados da pesquisa, o LangGraph parece ser uma estrutura para construir aplicativos mais complexos e com vários participantes usando modelos de linguagem grandes (LLMs). Alguns pontos-chave sobre o LangGraph:

- El